# AnchorFlow V5 — Piecewise Surface · TAR2000 · 15 epoch

Upload nguyên folder `AnchorFlow_v5_Piecewise` vào MyDrive. Mặc định một run V5 trên checkpoint V3 đã train 30 epoch; không stack ContextWide V4.

- 1.600 train / 400 val / 1.000 anonymous test; RGB+sparse+K tại inference.
- Metric teacher `D_cm/C_cm` chỉ dùng trong training. Không cần geometry TAR hay teacher weights.
- Khối V5 thêm 7.976 params / 0,202 G Conv MAC tại D4; latency GPU được đo sau run, không suy ra từ MAC.
- Objective V3 + fixed-GT boundary RMSE + directional barrier BCE. Đây là architecture+supervision bundle, không phải ablation chỉ kiến trúc.
- Các control tùy chọn chạy riêng bằng MODEL_NAME; Run all mặc định chỉ train V5 15 epoch. RMSE <0,7 chưa được chứng minh.


## 1. Mount và cấu hình

Chọn GPU rồi chạy từ trên xuống. `v5_piecewise` là run chính; `v5_transport` bỏ barrier; `v3` là control V3 với cùng boundary loss và thêm 15 epoch. Mỗi lựa chọn có run folder riêng.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

BUNDLE = Path("/content/drive/MyDrive/AnchorFlow_v5_Piecewise")
DRIVE_DATA = Path("/content/drive/MyDrive/GeoLift_Data")
DRIVE_RUNS = Path("/content/drive/MyDrive/GeoLift_RT_Runs")
WORK = Path("/content/anchorflow_v5_work")
CODE = Path("/content/anchorflow_v5_code")
MODEL_NAME = "v5_piecewise"  # optional: "v5_transport", "v3"
assert MODEL_NAME in ("v3", "v5_transport", "v5_piecewise")
RUN_NAMES = {"v5_piecewise": "AnchorFlow_v5_Piecewise_FT15", "v5_transport": "AnchorFlow_v5_Transport_FT15", "v3": "AnchorFlow_v3_BoundaryControl_FT15"}
RUN_TAG = ""  # e.g. "_arch_only" if changing loss coefficients
BOUNDARY_WEIGHT = 0.05
BARRIER_WEIGHT = 0.01
RUN_NAME = RUN_NAMES[MODEL_NAME] + RUN_TAG
EPOCHS = 15
BATCH_SIZE = 2
WORKERS = 2
RUN_EXPORT = True
assert BUNDLE.is_dir(), f"Upload folder to {BUNDLE}"
assert EPOCHS == 15
RUN_ROOT = DRIVE_RUNS / RUN_NAME
RUN_DIR = RUN_ROOT / "metric_kd"
print("Input:", DRIVE_DATA / "teacher_subset_2000")
print("Outputs:", RUN_DIR)
print("Colab SSD free GiB:", round(shutil.disk_usage("/content").free / 2**30, 1))

## 2. Kiểm SHA và copy code sang SSD Colab

Không copy dataset/teacher weights trong repo. Không xóa data/run cũ. Nếu SHA sai, upload lại toàn bộ folder; không bỏ qua gate. Notebook tự bỏ kiểm SHA chính nó vì Colab có thể lưu metadata.


In [ ]:
manifest = json.loads((BUNDLE / "bundle_manifest.json").read_text(encoding="utf-8"))
assert manifest["default_epochs"] == 15
assert manifest["default_variants"] == ["metric_kd"]
assert manifest["contains_data"] is False and manifest["contains_parent_student_checkpoint"] is True
CODE.mkdir(parents=True, exist_ok=True)
for name, expected in manifest["files"].items():
    assert Path(name).name == name, name
    if name.endswith(".ipynb"):
        continue
    source = BUNDLE / name
    raw = source.read_bytes()
    assert hashlib.sha256(raw).hexdigest() == expected, f"Bundle mismatch: {name}"
    destination = CODE / name
    if source.resolve() != destination.resolve():
        shutil.copy2(source, destination)
shutil.copy2(BUNDLE / "bundle_manifest.json", CODE / "bundle_manifest.json")
os.chdir(CODE)
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements.txt")], check=True)
import torch, timm
assert torch.cuda.is_available(), "Runtime > Change runtime type > GPU"
assert tuple(int(x) for x in torch.__version__.split("+")[0].split(".")[:2]) >= (2, 4)
print("GPU:", torch.cuda.get_device_name(0), "| torch:", torch.__version__, "| timm:", timm.__version__)

## 3. Resolve config và snapshot nguồn

`encoder_pretrained=false` vì nạp đầy đủ checkpoint V3 đã pretrained + fine-tune, không phải random init. Parent epoch14/cumulative29; V5 train thêm epoch0–14/cumulative30–44.

Peak LR 1e-4, encoder 0,25×; warm-up1 epoch rồi cosine. GT-boundary RMSE weight0,05 dùng mask GT cố định; barrier BCE weight0,01 chỉ ở full V5. Đặt BARRIER_WEIGHT=0 nếu muốn kiểm architecture-only với cùng objective. Đổi loss phải dùng RUN_TAG mới để tránh resume nhầm.


In [ ]:
cfg = json.loads((CODE / "config.json").read_text(encoding="utf-8"))
cfg.update(drive_data=str(DRIVE_DATA), drive_runs=str(DRIVE_RUNS), work=str(WORK),
           run_name=RUN_NAME, model_name=MODEL_NAME, epochs=EPOCHS, batch_size=BATCH_SIZE, workers=WORKERS,
           init_checkpoint=str(CODE / "init_v3_best.pth"))
cfg["architecture"] = {"v5_piecewise": "AnchorFlow-v5-PiecewiseSurface", "v5_transport": "AnchorFlow-v5-TransportOnly", "v3": "AnchorFlow-Edge-v3-RefineKD"}[MODEL_NAME]
cfg.update(boundary_weight=BOUNDARY_WEIGHT, barrier_weight=BARRIER_WEIGHT)
assert cfg["parent_cumulative_epoch"] == 29
assert cfg["new_lr_ratio"] == 1.0
assert cfg["parent_epoch"] == 14 and cfg["teacher_enabled"] is True
assert cfg["flow_steps"] == 3 and cfg["encoder_pretrained"] is False
assert hashlib.sha256(Path(cfg["init_checkpoint"]).read_bytes()).hexdigest() == cfg["init_sha256"]
CONFIG = CODE / "resolved_config.json"
CONFIG.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
SUBSET = DRIVE_DATA / "teacher_subset_2000"
for name in ("selected_2000_ids.json", "kitti_trainval_2000.tar", cfg["metric_tar"]):
    assert (SUBSET / name).is_file(), f"Thiếu {SUBSET / name}"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
snapshot = RUN_ROOT / "source_bundle"
snapshot.mkdir(parents=True, exist_ok=True)
for name in manifest["files"]:
    if name.endswith(".ipynb"):
        continue
    source = CODE / name
    target = snapshot / name
    if target.exists():
        assert hashlib.sha256(target.read_bytes()).hexdigest() == hashlib.sha256(source.read_bytes()).hexdigest(), f"Run này có source khác: {name}; dùng RUN_NAME mới"
    else:
        shutil.copy2(source, target)
shutil.copy2(BUNDLE / "bundle_manifest.json", snapshot / "bundle_manifest.json")
print(CONFIG.read_text())

def command(action, extra=()):
    subprocess.run([sys.executable, "-u", str(CODE / "run.py"), action,
                    "--config", str(CONFIG), "--variant", "metric_kd", *extra],
                   cwd=CODE, check=True)

## 4. Contract tests và data gates

Kiểm đúng selected2.000, train/val disjoint ID + raw drive. Inspect đủ2.000 metric records nhưng chỉ cache1.600 train. KD dùng C_cm>=0,5, không ghi đè GT hoặc bất kỳ sparse gốc nào (kể cả holdout). Val/test không có teacher.


In [ ]:
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(CODE), "-p", "test*.py", "-v"],
               cwd=CODE, check=True)
command("prepare")
contract = json.loads((WORK / "data_contract.json").read_text())
assert (contract["train"], contract["val"], contract["test"]) == (1600, 400, 1000)
assert contract["validation_teacher_used"] is False
assert contract["effective_kd_confidence_threshold"] == cfg["kd_conf_min"]
assert contract["min_non_gt_teacher_pixels"] >= cfg["min_teacher_pixels"]
shutil.copy2(WORK / "data_contract.json", RUN_ROOT / "data_contract.json")
print(json.dumps(contract, indent=2))

## 5. Inspect train/val

KITTI GT và metric teacher là hai nguồn khác nhau. Teacher có thể đã dùng GT khi generate; không gọi teacher-vs-GT là kết quả đánh giá độc lập.


In [ ]:
from data import KITTIDataset
import matplotlib.pyplot as plt
import numpy as np

train_data = KITTIDataset(cfg, "train", teacher=True)
val_data = KITTIDataset(cfg, "val", teacher=False)
assert len(train_data) == 1600 and len(val_data) == 400
train_sample, val_sample = train_data[0], val_data[0]
assert "teacher" not in val_sample and "confidence" not in val_sample
for label, sample in (("train", train_sample), ("val", val_sample)):
    names = ["RGB", "Sparse", "GT"] + (["Metric teacher", "C_cm"] if "teacher" in sample else [])
    images = [sample["rgb"].permute(1, 2, 0).numpy(), sample["sparse"][0].numpy(), sample["gt"][0].numpy()]
    if "teacher" in sample:
        images += [sample["teacher"][0].numpy(), sample["confidence"][0].numpy()]
    fig, axes = plt.subplots(1, len(images), figsize=(5*len(images), 3))
    for ax, name, values in zip(axes, names, images):
        if name == "RGB":
            ax.imshow(values)
        else:
            ax.imshow(np.ma.masked_where(values <= 0, values), cmap="turbo", vmin=0, vmax=1 if name == "C_cm" else 80)
        ax.set_title(name); ax.axis("off")
    fig.suptitle(label + ": " + sample["sid"])
    plt.show()
del train_data, val_data, train_sample, val_sample

## 6. Real-data smoke và no-op warm-start

Nạp đủ697 tensor/572.017 parent params; chỉ thiếu surface.*. D_full phải khớp V3 trước fine-tune. Gate kiểm finite loss/gradients, sensor fusion, teacher KD coverage thực tế.


In [ ]:
command("smoke")
smoke = json.loads((WORK / "smoke_report.json").read_text())
assert smoke["passed"] and smoke["migration"]["old_keys_dropped"] == 0
assert smoke["migration"]["parent_parameters_loaded"] == 572017
shutil.copy2(WORK / "smoke_report.json", RUN_ROOT / "smoke_report.json")
print(json.dumps(smoke, indent=2))
torch.cuda.empty_cache()

## 7. Train thêm 15 epoch

Đo400 val trước train (epoch-1) rồi epoch0–14. Đồng bộ best/last/log lên Drive sau mỗi epoch; resume khôi phục optimizer/scaler/RNG và kiểm source/protocol. Epoch-1 vẫn có thể là best nếu V5 không tiến bộ.

Không tự chạy control; muốn ablation thì đổi MODEL_NAME và chạy notebook riêng sau.


In [ ]:
command("train")
print("Train log:", RUN_DIR / "train_log.csv")
print("Best:", RUN_DIR / "best.pth")
print("Last/resume:", RUN_DIR / "last.pth")

## 8. Accuracy và observed GT-boundary bands

Primary vẫn global pixel RMSE trên25.424.992 valid GT pixels. Log soft/pre/hard, range, RGB-edge cũ và GT-discontinuity bands mới.

GT edge: hai GT kề nhau hợp lệ, jump > max(1m,5% min depth); band dùng khoảng cách Chebyshev 1/2/3/5/10px. Không xem mask GT sparse là nhãn occlusion hoàn hảo. Archive V4 không có metric này nên không điền số giả; audit CPU400val có trong bundle.


In [ ]:
command("evaluate")
import pandas as pd

parent = json.loads((CODE / "parent_val_metrics.json").read_text())
v4 = json.loads((CODE / "v4_val_metrics.json").read_text())
initial = json.loads((RUN_DIR / "initial_val_metrics.json").read_text())
best = json.loads((RUN_DIR / "val_metrics.json").read_text())
run_manifest = json.loads((RUN_DIR / "run_manifest.json").read_text())
entries = [
    ("v3 parent, completed 30 epochs (A100 FP16)", parent, "final"),
    ("v4 historical best, architecture+extra training", v4, "final"),
    (MODEL_NAME + " initial, same runtime", initial, "final"),
    (MODEL_NAME + " best soft", best, "final"),
    (MODEL_NAME + " best pre-fusion", best, "pre_anchor"),
    (MODEL_NAME + " best legacy hard", best, "legacy_hard_anchor"),
]
for other_name in RUN_NAMES:
    if other_name == MODEL_NAME:
        continue
    other_dir = DRIVE_RUNS / (RUN_NAMES[other_name] + RUN_TAG) / "metric_kd"
    if (other_dir / "val_metrics.json").is_file() and (other_dir / "run_manifest.json").is_file():
        other_manifest = json.loads((other_dir / "run_manifest.json").read_text())
        assert other_manifest["recipe_sha256"] == run_manifest["recipe_sha256"], "Recipe/data differs"
        other_metrics = json.loads((other_dir / "val_metrics.json").read_text())
        entries.append((other_name + " same budget (active barrier term differs)", other_metrics, "final"))
print("Historical V4 is not a matched-loss V5 control. Optional controls run separately.")
rows = []
for label, report, policy in entries:
    values = report[policy]
    score = values["all"]
    assert score["pixels"] == 25424992, "GT support changed"
    rows.append({"model/output": label, "RMSE m": score["rmse_m"], "MAE m": score["mae_m"],
                 "iRMSE km^-1": score["irmse_km_inv"], "edge RMSE m": values["edge"]["rmse_m"],
                 **{f"RMSE {key} m": values[key]["rmse_m"] for key in ("0-20","20-40","40-60","60-80","80-120")}})
comparison = pd.DataFrame(rows)
comparison.to_csv(RUN_DIR / "comparison_accuracy.csv", index=False)
display(comparison)
print("Best new epoch:", best["checkpoint_epoch"], "(-1 = no improvement over initialization)")
print("Same-runtime improvement:", initial["final"]["all"]["rmse_m"] - best["final"]["all"]["rmse_m"], "m")
print("Reached <0.7 m:", best["final"]["all"]["rmse_m"] < .7)
display(pd.DataFrame(best["stage_native_gt_metrics"]).T)

boundary_rows = []
for name, report in (("initial v3", initial), (MODEL_NAME+" best", best)):
    for kind in ("bands","rings"):
        for radius, values in report["gt_boundary"][kind].items():
            boundary_rows.append({"model":name,"kind":kind,"radius_px":int(radius),**{k:v for k,v in values.items() if k!="bad_pixel_rates"},
                                  **{f"bad_{t}m_rate":v for t,v in values["bad_pixel_rates"].items()}})
boundary_table = pd.DataFrame(boundary_rows)
boundary_table.to_csv(RUN_DIR / "gt_boundary_metrics.csv", index=False)
display(boundary_table)
fig, ax = plt.subplots(figsize=(7,4))
for name, rows in boundary_table[boundary_table["kind"]=="bands"].groupby("model"):
    ax.plot(rows["radius_px"],rows["rmse_m"],marker="o",label=name)
ax.set(xlabel="Chebyshev distance to observed GT discontinuity (px)",ylabel="Cumulative band RMSE (m)")
ax.legend(); ax.grid(alpha=.25); fig.tight_layout()
fig.savefig(RUN_DIR/"gt_boundary_curve.png",dpi=160)
plt.show()


## 9. Curves và loss/module dynamics

Theo dõi global/far/boundary, weighted loss và surface correction. Scalar loss nhỏ không đủ chứng minh gradient nhỏ. Signed surface amplitude không phải confidence probability.


In [ ]:
log = pd.read_csv(RUN_DIR / "train_log.csv")
assert len(log) == 15 and log["epoch"].tolist() == list(range(15)), "Run chưa hoàn thành 15 epoch"
assert set(log["train_samples"]) == {1600} and set(log["val_samples"]) == {400}
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for key in ("val_rmse_m", "val_pre_anchor_rmse_m", "val_legacy_hard_rmse_m"):
    axes[0,0].plot(log["cumulative_epoch_index"], log[key], label=key)
axes[0,0].axhline(initial["final"]["all"]["rmse_m"], color="gray", linestyle="--", label="initial soft")
axes[0,0].set_ylabel("Global RMSE (m)"); axes[0,0].legend(fontsize=8)
for key in ("0-20","20-40","40-60","60-80"):
    axes[0,1].plot(log["epoch"], log[f"val_rmse_{key}_m"], label=key)
axes[0,1].set_ylabel("Range RMSE (m)"); axes[0,1].legend()
weighted = [key for key in log if key.startswith("loss_weighted_")]
for key in weighted:
    axes[1,0].plot(log["epoch"], log[key].clip(lower=1e-8), label=key.removeprefix("loss_weighted_"))
axes[1,0].set_yscale("log"); axes[1,0].set_ylabel("Weighted loss"); axes[1,0].legend(fontsize=8, ncol=2)
for key in ("loss_sensor_gate_mean","loss_abs_delta4_mean","loss_abs_delta1_mean","loss_surface_abs_delta_mean","loss_surface_amplitude"):
    axes[1,1].plot(log["epoch"], log[key], label=key.removeprefix("loss_"))
axes[1,1].legend(fontsize=8); axes[1,1].set_ylabel("Gate / correction(m) / signed amplitude")
for ax in axes.flat:
    ax.set_xlabel("Epoch (cumulative on top-left)"); ax.grid(alpha=.25)
fig.tight_layout()
fig.savefig(RUN_DIR / "training_diagnostics.png", dpi=160)
plt.show()
budget = log[["epoch", "loss_total", *weighted]].copy()
for key in weighted:
    budget[key + "_pct"] = 100 * budget[key] / budget["loss_total"]
budget.to_csv(RUN_DIR / "loss_budget.csv", index=False)
display(budget.tail(3))
display(log[["epoch","val_native_D4_base_rmse_m","val_native_D4_rmse_m",
             "val_gt_boundary_1px_rmse_m","val_gt_boundary_3px_rmse_m","val_gt_boundary_5px_rmse_m",
             "loss_barrier_support","loss_barrier_positive_fraction","loss_surface_barrier_mean"]].tail())


## 10. Anonymous test 1.000 ảnh

Dùng best checkpoint xuất PNG depth×256 và ZIP submission. Không có public GT, không có test RMSE. Nếu Drive chưa có test TAR, prepare lấy official test và cache về GeoLift_Data/test_1000/.


In [ ]:
command("test")
test_report = json.loads((RUN_DIR / "test_report.json").read_text())
assert test_report["samples"] == 1000 and test_report["anonymous_no_gt"] is True
print("Submission:", RUN_DIR / "kitti_test_predictions.zip")
print(json.dumps(test_report, indent=2))

## 11. Profile cùng GPU

Đo parent V3 và best V5 ở FP16/batch1/352×1216 trong subprocess riêng. Conv MAC không đếm shifts, interpolation, pooling, elementwise hoặc memory traffic. Component pass là instrumented timing, không cộng để thay total latency.


In [ ]:
command("profile", ("--parent",))
command("profile")
new_profile = json.loads((RUN_DIR / "profile.json").read_text())
old_profile = json.loads((RUN_DIR / "parent_profile_same_device.json").read_text())
keys = ("device","torch","precision","batch","shape","channels_last","compiled_total")
comparable = all(new_profile.get(key) == old_profile.get(key) for key in keys)
assert comparable, "Profile settings changed between parent and candidate"
efficiency = pd.DataFrame([{
    "model": name, "device": p["device"], "torch": p["torch"],
    "params": p["total_parameters"], "G Conv/Linear MAC": p["total_conv_linear_macs"]/1e9,
    "median ms": p["median_ms"], "P95 ms": p["p95_ms"], "peak MiB": p["peak_cuda_allocated_mib"]
} for name, p in (("v3 parent same device", old_profile), (MODEL_NAME + " best", new_profile))])
efficiency.to_csv(RUN_DIR / "comparison_efficiency.csv", index=False)
display(efficiency)
print("Measured median latency change %:", 100*(new_profile["median_ms"]/old_profile["median_ms"]-1))
components = pd.DataFrame({
    "parameters": new_profile["parameters"], "conv_linear_macs": new_profile["conv_linear_macs"],
    "instrumented_median_ms": new_profile["eager_component_median_ms"]
})
components.to_csv(RUN_DIR / "component_profile.csv")
display(components)

## 12. ONNX và summary

Export best static batch1/opset17/FP32, checker + CPU ONNX Runtime parity có/không sparse. Chưa phải TensorRT và chưa bảo đảm tốc độ trên thiết bị edge cụ thể.


In [ ]:
if RUN_EXPORT:
    command("export")
    export_report = json.loads((RUN_DIR / "export_report.json").read_text())
    assert export_report["onnxruntime_cpu_parity"] and not export_report["untrained_weights"]
    print(json.dumps(export_report, indent=2))
summary = {
    "model_name": MODEL_NAME, "recipe_sha256": run_manifest["recipe_sha256"], "architecture": cfg["architecture"], "parent_epoch": cfg["parent_epoch"],
    "experiment_type": "architecture + fixed-GT boundary supervision; barrier BCE only in full V5",
    "best_gt_boundary_metrics": best["gt_boundary"],
    "extra_epochs_completed": len(log), "best_new_epoch": best["checkpoint_epoch"],
    "initial_soft_rmse_m": initial["final"]["all"]["rmse_m"],
    "best_soft_rmse_m": best["final"]["all"]["rmse_m"],
    "best_pre_fusion_rmse_m": best["pre_anchor"]["all"]["rmse_m"],
    "best_legacy_hard_rmse_m": best["legacy_hard_anchor"]["all"]["rmse_m"],
    "goal_0_7_reached": best["final"]["all"]["rmse_m"] < .7,
    "params": new_profile["total_parameters"], "gpu": new_profile["device"],
    "median_ms": new_profile["median_ms"], "runtime_settings_comparable_to_parent": comparable,
    "teacher_train_only": True, "anonymous_test_has_public_gt": False,
}
(RUN_DIR / "experiment_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))
print("ALL OUTPUTS:", RUN_DIR)

## Output Drive

```text
MyDrive/GeoLift_RT_Runs/AnchorFlow_v5_Piecewise_FT15/
├── source_bundle/
├── data_contract.json / smoke_report.json
└── metric_kd/
    ├── best.pth / last.pth / run_manifest.json / resolved_config.json
    ├── initial_val_metrics.json / best_val_metrics.json / val_metrics.json
    ├── train_log.csv / train_log.jsonl / train.log
    ├── comparison_accuracy.csv / gt_boundary_metrics.csv
    ├── training_diagnostics.png / loss_budget.csv / gt_boundary_curve.png
    ├── parent_profile_same_device.json / profile.json / component_profile.csv
    ├── kitti_test_predictions.zip / test_report.json
    └── anchorflow_edge_fp32.onnx / export_report.json / experiment_summary.json
```

Control có folder riêng. So với V3 parent cũ còn lẫn hiệu ứng thêm15epoch; recipe hash chung không xóa được khác biệt active barrier supervision. Không kết luận novelty chỉ từ một run.
